<a href="https://colab.research.google.com/github/sasankgit/generativeai-imagegen/blob/main/facial_imagegen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q kagglehub
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/face_diffusion/samples',exist_ok=true)

In [ ]:
import math, glob, copy
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
import kagglehub

device = "cuda" if torch.cuda.is_available() else "cpu"
IMG = 64
BATCH = 64
EPOCHS = 100
LR = 2e-4
T = 1000
EMA_DECAY = 0.999
SAVE_DIR = "/content/drive/MyDrive/face_diffusion"
print("device:", device)

In [ ]:
path = kagglehub.dataset_download("ashwingupta3012/human-faces")
print("downloaded to:", path)

files = []
for ext in ("jpg", "jpeg", "png"):
    files += glob.glob(f"{path}/**/*.{ext}", recursive=True)
    files += glob.glob(f"{path}/**/*.{ext.upper()}", recursive=True)
print("images found:", len(files))

tfm = transforms.Compose([
    transforms.Resize(IMG),
    transforms.CenterCrop(IMG),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,) * 3, (0.5,) * 3),
])

class Faces(Dataset):
    def __init__(self, files):
        self.files = files
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        return tfm(Image.open(self.files[i]).convert("RGB"))

loader = DataLoader(Faces(files), batch_size=BATCH, shuffle=True,
                    num_workers=2, drop_last=True, pin_memory=True)

batch = next(iter(loader))
save_image(batch[:16] * 0.5 + 0.5, "real.png", nrow=4)
from IPython.display import Image as IPImage, display
display(IPImage("real.png"))

In [ ]:
class SinEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        f = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        a = t[:, None].float() * f[None]
        return torch.cat([a.sin(), a.cos()], -1)

class Block(nn.Module):
    def __init__(self, cin, cout, tdim):
        super().__init__()
        self.n1 = nn.GroupNorm(8, cin)
        self.c1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.t = nn.Linear(tdim, cout)
        self.n2 = nn.GroupNorm(8, cout)
        self.c2 = nn.Conv2d(cout, cout, 3, padding=1)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()
    def forward(self, x, t):
        h = self.c1(F.silu(self.n1(x)))
        h = h + self.t(t)[:, :, None, None]
        h = self.c2(F.silu(self.n2(h)))
        return h + self.skip(x)

class Attn(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n = nn.GroupNorm(8, c)
        self.a = nn.MultiheadAttention(c, 4, batch_first=True)
    def forward(self, x):
        B, C, H, W = x.shape
        h = self.n(x).flatten(2).transpose(1, 2)
        h, _ = self.a(h, h, h)
        return x + h.transpose(1, 2).reshape(B, C, H, W)

class UNet(nn.Module):
    def __init__(self, ch=(64, 128, 256, 256), tdim=256):
        super().__init__()
        self.temb = nn.Sequential(SinEmb(64), nn.Linear(64, tdim), nn.SiLU(), nn.Linear(tdim, tdim))
        self.inc = nn.Conv2d(3, ch[0], 3, padding=1)
        n = len(ch)

        self.down = nn.ModuleList()
        cin = ch[0]
        for i in range(n):
            blk = Block(cin, ch[i], tdim)
            att = Attn(ch[i]) if i >= 2 else nn.Identity()
            dn = nn.Conv2d(ch[i], ch[i], 3, stride=2, padding=1) if i < n - 1 else nn.Identity()
            self.down.append(nn.ModuleList([blk, att, dn]))
            cin = ch[i]

        self.mid1 = Block(ch[-1], ch[-1], tdim)
        self.mida = Attn(ch[-1])
        self.mid2 = Block(ch[-1], ch[-1], tdim)

        self.up = nn.ModuleList()
        prev = ch[-1]
        for i in reversed(range(n)):
            blk = Block(prev + ch[i], ch[i], tdim)
            att = Attn(ch[i]) if i >= 2 else nn.Identity()
            upl = nn.Sequential(nn.Upsample(scale_factor=2, mode="nearest"),
                                nn.Conv2d(ch[i], ch[i], 3, padding=1)) if i > 0 else nn.Identity()
            self.up.append(nn.ModuleList([blk, att, upl]))
            prev = ch[i]

        self.out = nn.Sequential(nn.GroupNorm(8, ch[0]), nn.SiLU(), nn.Conv2d(ch[0], 3, 3, padding=1))

    def forward(self, x, t):
        t = self.temb(t)
        h = self.inc(x)
        skips = []
        for blk, att, dn in self.down:
            h = att(blk(h, t))
            skips.append(h)
            h = dn(h)
        h = self.mid2(self.mida(self.mid1(h, t)), t)
        for blk, att, upl in self.up:
            h = torch.cat([h, skips.pop()], 1)
            h = att(blk(h, t))
            h = upl(h)
        return self.out(h)

model = UNet().to(device)
print("params (M):", sum(p.numel() for p in model.parameters()) / 1e6)

In [ ]:
betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1 - betas
abar = torch.cumprod(alphas, 0)

def q_sample(x0, t, noise):
    a = abar[t][:, None, None, None]
    return a.sqrt() * x0 + (1 - a).sqrt() * noise

@torch.no_grad()
def ddpm_sample(net, n=16):
    net.eval()
    x = torch.randn(n, 3, IMG, IMG, device=device)
    for i in reversed(range(T)):
        t = torch.full((n,), i, device=device, dtype=torch.long)
        eps = net(x, t)
        mean = (x - (1 - alphas[i]) / (1 - abar[i]).sqrt() * eps) / alphas[i].sqrt()
        x = mean + betas[i].sqrt() * torch.randn_like(x) if i > 0 else mean
    return x.clamp(-1, 1)

@torch.no_grad()
def ddim_sample(net, n=16, steps=50):
    net.eval()
    ts = torch.linspace(T - 1, 0, steps).long().tolist()
    x = torch.randn(n, 3, IMG, IMG, device=device)
    for j, i in enumerate(ts):
        t = torch.full((n,), i, device=device, dtype=torch.long)
        eps = net(x, t)
        ab = abar[i]
        x0 = ((x - (1 - ab).sqrt() * eps) / ab.sqrt()).clamp(-1, 1)
        ab_prev = abar[ts[j + 1]] if j + 1 < len(ts) else torch.tensor(1.0, device=device)
        x = ab_prev.sqrt() * x0 + (1 - ab_prev).sqrt() * eps
    return x.clamp(-1, 1)

In [ ]:
ema = copy.deepcopy(model).eval()
opt = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler("cuda")
start = 0
ckpt_path = f"{SAVE_DIR}/ckpt.pt"

if os.path.exists(ckpt_path):
    ck = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ck["model"])
    ema.load_state_dict(ck["ema"])
    opt.load_state_dict(ck["opt"])
    start = ck["epoch"] + 1
    print("resumed from epoch", start)

for epoch in range(start, EPOCHS):
    model.train()
    total = 0
    for x in loader:
        x = x.to(device, non_blocking=True)
        t = torch.randint(0, T, (x.size(0),), device=device)
        noise = torch.randn_like(x)

        with torch.amp.autocast("cuda", dtype=torch.float16):
            loss = F.mse_loss(model(q_sample(x, t, noise), t), noise)

        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()

        with torch.no_grad():
            for pe, pm in zip(ema.parameters(), model.parameters()):
                pe.lerp_(pm, 1 - EMA_DECAY)

        total += loss.item()
    print(f"epoch {epoch+1}/{EPOCHS}  loss {total/len(loader):.4f}")

    if (epoch + 1) % 5 == 0 or epoch == 0:
        imgs = ddim_sample(ema, 16) * 0.5 + 0.5
        save_image(imgs, f"{SAVE_DIR}/samples/epoch_{epoch+1:03d}.png", nrow=4)
        torch.save({"model": model.state_dict(), "ema": ema.state_dict(),
                    "opt": opt.state_dict(), "epoch": epoch}, ckpt_path)

In [ ]:
ck = torch.load(ckpt_path, map_location=device)
ema.load_state_dict(ck["ema"])
imgs = ddim_sample(ema, 16, steps=100) * 0.5 + 0.5
save_image(imgs, "generated.png", nrow=4)
display(IPImage("generated.png"))